# Brain-state transitions vs. anxiety and depression

Figures on Cam-CAN HADS (anxiety and depression categories):

1. **Four rankings side by side.** `bstm_selection` (transition matrices) and
   `fcm_selection` (static connectivity) at rest and during the movie, with **every row**
   of their `summary.csv`: every model, arm, atlas, window and state set (`FIG1_MODELS`,
   `FIG1_ARMS` narrow it). One block of panels per model. Every row is plotted as its **gain over
   its own model's covariates-only baseline** (Age + Sex, fitted here): every row already carries
   the covariates, so its raw score mostly reflects how well its model fits Age + Sex. On the
   gain axis, 0 is "no better than covariates" for every model and the models compare directly.
   A row whose gain clears its baseline's sd at rest **and** in the movie is drawn in colour,
   ± sd over seeds, in both task columns and named in the legend (the best `HIGHLIGHT_MAX`; the
   table lists all). The best fcm row of each task is marked (● rest, ◆ movie): filled where it
   won, hollow where carried over.

   **1.2–1.6: what drives the score.** The same rows, grouped by one axis of the grid at a
   time: window size, atlas, model, state method, arm (`FACTORS`).
2. **Volcano, bstm only.** For rest and movie, the `TOP_N` best state sets of the `MODEL` /
   `ARM` ranking. Every transition cell is tested against HADS with an effect size on x
   (`X_AXIS`), adjusted for `COVARIATES`. Colour shows the state set, and the legend rank (#1…)
   is the set's position in that panel's ranking.

The selection runs (`fmri-decomp select-bstm` / `select-fcm`) produce the rankings.
This notebook only reads them, plus the `transitions/` tables, and writes nothing
to `outputs/`. Everything you might want to change is in the next cell.

In [ ]:
import os
from pathlib import Path

ROOT = Path(os.environ.get("FMRIDECOMP_OUTPUTS",
                           "/project/6008063/tamires/DecomposingfMRI/outputs"))
CAMCAN = Path(os.environ.get("CAMCAN_PHENO",
                             "/project/6008063/tamires/cohorts/camcan/dataman/useraccess/"
                             "opendata/paule_toussaint_camcan01870"))
PHENO_FILES = [(CAMCAN / "approved_data.tsv", "\t"),   # HADS
               (CAMCAN / "standard_data.csv", ",")]     # Age, Sex
PHENO_ID = "CCID"

# label -> column. Labels are what the figures say.
TARGETS = {"anxiety":    "additional_HADS_anx_category",
           "depression": "additional_HADS_dep_category"}
ORDINAL_LEVELS = ["Normal", "Mild", "Moderate", "Severe"]   # low -> high

# task label in the *_selection trees -> task value inside the transition tables
TASKS = {"rest": "rest", "movie": "movie"}                  # matched case-insensitively
COHORTS = ["camcan", "camcan_rest"]                         # cohort= dirs to read

MODEL = "ridge"                                  # figure 2 picks its sets from this model's ranking
ARM = {"bstm": "cells [joint]", "fcm": "edges"}  # ... and this arm (figure 1 shows every arm)

TOP_N = 10                                       # state sets per volcano panel
COVARIATES = ["Age", "Sex", "n_transitions"]
CATEGORICAL = ["Sex"]
ALPHA = 0.05                                     # BH-FDR, per panel
SAVE_DIR = None                                  # e.g. Path("figures/08") to save PNG + SVG

# Figure 1 shows every row of the four summary.csv files, narrowed by these.
FIG1_MODELS = None                               # None = all but logistic (AUC, another scale)
FIG1_ARMS = None                                 # None = every arm, e.g. ["cells [joint]", "edges"]

# The dashed lines: one covariates-only model per (phenotype, model), fitted HERE
# on BASELINE_COVARIATES with the pipeline's own CV (score_once), on everyone the
# bstm runs scored. The same lines in all four columns.
BASELINE_COVARIATES = ["Age", "Sex"]
BASELINE_ARM = "(covariates only)"               # how the runs label their own baseline rows
SEEDS = [0, 1, 2, 3, 4]                          # as select-bstm / select-fcm
MARGIN_SD = 0                                    # "clears" = gain - MARGIN_SD * row sd > baseline sd
HIGHLIGHT_MAX = 10                               # rows named per panel (best worst-task gain); the table has all
REPO = Path.cwd().resolve().parent               # for fmri_decomposition.score_once
N_JOBS = (len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity")
          else os.cpu_count())                   # cores for the baseline fits (your SLURM share)

# Figures 1.2-1.6: one per axis of the grid. column -> title. Also available: "K", "embedding".
FACTORS = {"window": "window size", "atlas": "atlas", "model": "model",
           "method": "state method", "arm": "arm"}
FACTOR_TREES = ["bstm"]                          # add "fcm" to pool its rows where the axis exists

# Figure 2's x axis. All three adjust for COVARIATES by regression.
#   "log2fc"  log2( mean cell, CASE_FROM and above / mean cell, Normal ), with a pseudocount
#   "slope"   change in the cell's probability per HADS level (raw units)
#   "r"       partial Spearman r (p is then a function of |r|: the narrow V)
X_AXIS = "log2fc"
CASE_FROM = "Mild"                               # log2fc: this level and above = elevated
SHRINK = 1.0                                     # log2fc pseudocount, in units of the average cell:
                                                 #   0 = raw ratio (widest), 1 = moderate, 5+ = narrow
MIN_PRESENT = 0.10                               # drop cells non-zero in < this share of subjects
LABEL_TOP = 3                                    # cells annotated per volcano panel

In [ ]:
import sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy import stats

INK, INK2, MUTED, SURFACE = "#0b0b0b", "#52514e", "#a3a29d", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK2, "axes.titlesize": 11,
    "axes.titlecolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "font.size": 9, "legend.frameon": False})

def hive(path, key):
    """One `key=value` partition from a path, by name."""
    return next((p.split("=", 1)[1] for p in Path(path).parts if p.startswith(f"{key}=")), None)

def fmt_window(w):
    return str(int(w)) if float(w).is_integer() else str(w)

def set_label(r):
    """A readable name for one summary row: state set or static-FC atlas."""
    if r.get("tree") == "fcm":
        return r["atlas"]
    w = "1 TR" if float(r["window_s"]) < 0 else f"{fmt_window(r['window_s'])}s"
    return f"{r['states']} · {r['atlas']} · {w}"

def save(fig, name):
    if SAVE_DIR:
        Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
        for ext in ("png", "svg"):
            fig.savefig(Path(SAVE_DIR) / f"{name}.{ext}", dpi=200, bbox_inches="tight")

def design(cov, intercept=True):
    """Covariates -> numeric matrix: categoricals one-hot, gaps mean-filled."""
    X = pd.get_dummies(cov, columns=[c for c in CATEGORICAL if c in cov],
                       drop_first=True).astype(float)
    X = X.fillna(X.mean()).to_numpy()
    return np.column_stack([np.ones(len(X)), X]) if intercept else X
TREE_COLOUR = {"bstm": "#2a78d6", "fcm": "#eb6834"}
TASK_COLOUR = dict(zip(TASKS, ["#1baf7a", "#6250d6"]))    # figures 1.2-1.6
TASK_MARKER = dict(zip(TASKS, ["o", "D"]))
DASHES = ["--", "-.", ":", (0, (6, 2, 1, 2))]             # one per model's baseline
# 8 validated hues, then two neutrals for slots 9-10. The marker shape is the
# second encoding, so no two sets rely on colour alone.
SET_COLOURS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4",
               "#008300", "#6250d6", "#e34948", "#52514e", "#8c6d4f"]
SET_MARKERS = ["o", "s", "^", "D", "v"]

In [ ]:
def read_summaries(tree):
    files = sorted((ROOT / f"{tree}_selection").glob("task=*/target=*/summary.csv"))
    if not files:
        raise SystemExit(f"no summary.csv under {ROOT / f'{tree}_selection'}")
    return pd.concat([pd.read_csv(p).assign(tree=tree, task=hive(p, "task"),
                                            target=hive(p, "target"))
                      for p in files], ignore_index=True)

def read_pheno(covariates):
    frames = [pd.read_csv(p, sep=s, dtype={PHENO_ID: str}) for p, s in PHENO_FILES]
    ph = frames[0]
    for f in frames[1:]:
        ph = ph.merge(f.drop(columns=[c for c in f if c in ph and c != PHENO_ID]),
                      on=PHENO_ID, how="outer")
    lower = {c.lower(): c for c in ph.columns}              # Age vs age
    ph = ph.rename(columns={lower[c.lower()]: c for c in covariates if c.lower() in lower})
    ph["sub"] = ph[PHENO_ID].astype(str).str.strip().str.upper()
    code = {lvl: k for k, lvl in enumerate(ORDINAL_LEVELS)}
    for lab, col in TARGETS.items():
        ph[lab] = ph[col].astype("string").str.strip().map(code).astype(float)
    for c in set(covariates) - set(CATEGORICAL) - {"n_transitions"}:
        ph[c] = pd.to_numeric(ph[c], errors="coerce")
    return ph[["sub", *TARGETS, *(c for c in covariates if c in ph)]]

def load_set(atlas, window_s, states, task):
    """One state set's transition table for one task, one row per subject."""
    base = (ROOT / "transitions" / f"atlas={atlas}" / f"window_s={fmt_window(window_s)}"
            / f"states={states}")
    files = [p for p in base.rglob("subjects.parquet")
             if hive(p, "cohort") in COHORTS
             and (hive(p, "task") or task).lower() == TASKS[task].lower()]
    if not files:
        raise SystemExit(f"no transition table for {base} task={task}")
    d = pd.concat(map(pd.read_parquet, files), ignore_index=True)
    d = d[d["task"].astype(str).str.lower() == TASKS[task].lower()]
    d["sub"] = d["sub"].astype(str).str.strip().str.upper()
    if d["sub"].duplicated().any():
        raise SystemExit(f"{states} {task}: a subject appears twice -- narrow COHORTS")
    return d

def cell_cols(d):
    return [c for c in d.columns if "->" in c]

def as_p_cond(d, K):
    """p_joint -> Markov p(j | i): back to counts, then renormalise each row."""
    cols = cell_cols(d)
    M = d[cols].to_numpy(float).reshape(len(d), K, K) * d["n_transitions"].to_numpy(float)[:, None, None]
    s = M.sum(2, keepdims=True)
    return pd.DataFrame(np.divide(M, s, out=np.zeros_like(M), where=s > 0).reshape(len(d), -1),
                        columns=cols, index=d.index)

def partial_spearman(A, y, cov):
    """Rank everything, residualise on the covariates, correlate. Returns r, p."""
    X = design(cov)
    R = np.column_stack([stats.rankdata(y), np.apply_along_axis(stats.rankdata, 0, A)])
    R = R - X @ np.linalg.lstsq(X, R, rcond=None)[0]
    R = (R - R.mean(0)) / np.where(R.std(0) > 0, R.std(0), np.nan)
    r = R[:, 1:].T @ R[:, 0] / len(R)
    df = len(R) - X.shape[1] - 1
    p = 2 * stats.t.sf(np.abs(r * np.sqrt(df / (1 - r ** 2))), df)
    return r, p

def top_sets(task, lab, n=TOP_N):
    """The n best bstm state sets of one ranking, as summary.csv rows."""
    g = summ[(summ["tree"] == "bstm") & (summ["task"] == task) & (summ["phenotype"] == lab)
             & (summ["arm"] == ARM["bstm"]) & (summ["model"] == MODEL)]
    return (g.sort_values("mean", ascending=False)
             .drop_duplicates(["atlas", "window_s", "states"]).head(n).reset_index(drop=True))

def merged(s, task, lab):
    """One state set's table joined to phenotype, cells as ARM asks for."""
    t = load_set(s["atlas"], s["window_s"], s["states"], task)
    if "[cond]" in ARM["bstm"]:
        t[cell_cols(t)] = as_p_cond(t, int(s["K"]))
    d = t.merge(pheno, on="sub", suffixes=("", "_pheno")).dropna(
        subset=[lab, *(c for c in COVARIATES if c not in CATEGORICAL)])
    return d, cell_cols(t)

label_of = {col: lab for lab, col in TARGETS.items()}
summ = pd.concat([read_summaries("bstm"), read_summaries("fcm")], ignore_index=True)
summ = summ[summ["target"].isin(label_of) & summ["task"].isin(TASKS)].assign(phenotype=lambda d: d["target"].map(label_of))
summ["label"] = summ.apply(set_label, axis=1)
summ["window"] = summ["window_s"].map(lambda w: None if pd.isna(w) else "1 TR" if w < 0 else f"{w:g}s")
summ[["method", "embedding", "K_"]] = summ["states"].str.rsplit("_", n=2, expand=True).reindex(columns=range(3))

pheno = read_pheno(COVARIATES + BASELINE_COVARIATES)
print(f"phenotype: {len(pheno):,} subjects;",
      ", ".join(f"{lab} {pheno[lab].value_counts().sort_index().astype(int).to_dict()}"
                for lab in TARGETS))

# Check this before reading a gap as a result: the runs are separate, so `n` can differ.
summ.pivot_table(index=["tree", "task"], columns="phenotype", values="n", aggfunc="max")

## 1. The four rankings

In [ ]:
import time
from joblib import Parallel, delayed

models = FIG1_MODELS or sorted(set(summ["model"]) - {"logistic"})
rows = summ[(summ["arm"] != BASELINE_ARM) & summ["model"].isin(models)
            & (summ["arm"].isin(FIG1_ARMS) if FIG1_ARMS else True)]
# one name per row that is the SAME row at rest and in the movie
rows = rows.assign(entity=rows["tree"] + " · " + rows["label"] + " · " + rows["arm"] + " · " + rows["model"])

def _fit(repo, X, y, model, seed):
    """One baseline score on ONE core. hgb otherwise spreads every tiny fit over all of a
    node's cores, which on a cluster is far slower than one core per fit side by side."""
    import sys
    from threadpoolctl import threadpool_limits
    sys.path.insert(0, repo)
    from fmri_decomposition.bstm_selection import score_once
    with threadpool_limits(1):
        return score_once(X, y, model, seed)

def baselines():
    """{phenotype: {model: mean, std, n}} -- every (phenotype, model, seed) fit in parallel."""
    data = {}
    for lab in TARGETS:                               # everyone the bstm runs scored
        subs = set().union(*(load_set(s["atlas"], s["window_s"], s["states"], task)["sub"]
                             for task in TASKS for _, s in top_sets(task, lab, 1).iterrows()))
        d = pheno[pheno["sub"].isin(subs)].dropna(
            subset=[lab, *(c for c in BASELINE_COVARIATES if c not in CATEGORICAL)])
        data[lab] = design(d[BASELINE_COVARIATES], intercept=False), d[lab].to_numpy(float)
    jobs = [(lab, m, s) for lab in TARGETS for m in models for s in SEEDS]
    scores = Parallel(n_jobs=min(N_JOBS, len(jobs)))(
        delayed(_fit)(str(REPO), *data[lab], m, s) for lab, m, s in jobs)
    out = {lab: {m: [] for m in models} for lab in TARGETS}
    for (lab, m, _), sc in zip(jobs, scores):
        out[lab][m].append(sc)
    return {lab: {m: {"mean": np.nanmean(v), "std": np.nanstd(v), "n": len(data[lab][1])}
                  for m, v in out[lab].items()} for lab in TARGETS}

# Its own cell, so re-drawing a figure does not refit.
t0 = time.time()
base = baselines()
print(f"{len(TARGETS) * len(models) * len(SEEDS)} baseline fits on {N_JOBS} core(s) in {time.time() - t0:.0f}s")
pd.DataFrame({(lab, m): b for lab in base for m, b in base[lab].items()}).T.round(3)

In [ ]:
RUNS = [(task, tree) for task in TASKS for tree in ("bstm", "fcm")]
ORIGIN_MARKER = dict(zip(TASKS, ["o", "D", "s", "^"]))   # which task the fcm best came from

# Every row against ITS OWN model's baseline: gain 0 = no better than Age + Sex with that model.
_key = list(zip(rows["phenotype"], rows["model"]))
rows = rows.assign(gain=rows["mean"] - [base[l][m]["mean"] for l, m in _key],
                   base_sd=[base[l][m]["std"] for l, m in _key])

def draw_zero(ax, sd):
    """The baseline on the gain axis: 0, with its own ± sd over seeds."""
    ax.axhspan(-sd, sd, color=MUTED, alpha=.18, lw=0)
    ax.axhline(0, color=INK2, ls="--", lw=.9)

def above_both(d):
    """Rows whose gain clears their baseline's sd in EVERY task, best worst-task gain first."""
    ok = d["gain"] - MARGIN_SD * d["std"] > d["base_sd"]
    worst = d[ok].groupby("entity").agg(tasks=("task", "nunique"), worst=("gain", "min"))
    return worst[worst["tasks"] == len(TASKS)].sort_values("worst", ascending=False).index.tolist()

def fcm_best(d):
    """{task: the fcm row with the largest gain in that task}."""
    f = d[d["tree"] == "fcm"]
    return {task: g.loc[g["gain"].idxmax()] for task, g in f.groupby("task", sort=False)}

def plot_runs(ax, d, sd):
    rng = np.random.default_rng(0)
    keep = above_both(d)
    shown = keep[:HIGHLIGHT_MAX]
    style = {e: (SET_COLOURS[k % len(SET_COLOURS)], SET_MARKERS[k // len(SET_COLOURS) % len(SET_MARKERS)])
             for k, e in enumerate(shown)}
    offset = dict(zip(shown, np.linspace(-.3, .3, len(shown)) if len(shown) > 1 else [0]))
    best, orange = fcm_best(d), TREE_COLOUR["fcm"]
    draw_zero(ax, sd)
    for i, (task, tree) in enumerate(RUNS):
        g = d[(d["task"] == task) & (d["tree"] == tree)]
        ax.scatter(i + rng.uniform(-.14, .14, len(g)), g["gain"], s=10,
                   color=TREE_COLOUR[tree], alpha=.2, lw=0)
        for r in g[g["entity"].isin(style)].itertuples():       # same x offset in both tasks
            c, m = style[r.entity]
            ax.errorbar(i + offset[r.entity], r.gain, yerr=r.std, fmt=m, ms=6, lw=1.2,
                        color=c, mec=SURFACE, mew=.8, zorder=3)
        if tree == "fcm":                                       # filled where it won, hollow carried over
            for origin, w in best.items():
                r = g[g["entity"] == w["entity"]]
                if r.empty:
                    continue
                own = origin == task
                ax.errorbar(i if own else i + .22, r["gain"].iloc[0], yerr=r["std"].iloc[0],
                            fmt=ORIGIN_MARKER[origin], ms=8, lw=1.5, color=orange, zorder=4,
                            mfc=orange if own else SURFACE, mec=SURFACE if own else orange, mew=1.5)
    ax.set_xticks(range(len(RUNS)), [f"{tree}\n{task}" for task, tree in RUNS])
    for k in range(1, len(TASKS)):
        ax.axvline(2 * k - .5, color=MUTED, lw=.6)
    ax.set_xlim(-.6, len(RUNS) - .4)

    handles = [Line2D([], [], ls="none", marker=style[e][1], ms=6, color=style[e][0], label=e)
               for e in shown]
    if len(keep) > len(shown):
        handles.append(Line2D([], [], ls="none", label=f"… and {len(keep) - len(shown)} more (table below)"))
    handles += [Line2D([], [], ls="none", marker=ORIGIN_MARKER[o], ms=7, color=orange,
                       label=f"fcm best at {o}:  {w['label']} · {w['arm']}  (filled where it won)")
                for o, w in best.items()]
    handles.append(Patch(color=MUTED, alpha=.35, label=f"Age + Sex baseline ± sd  (0 on this axis, sd {sd:.3f})"))
    return handles, f"above the baseline at {' AND '.join(TASKS)}:  {len(keep)} row(s)"

# One block per model: its panels on top, their legends underneath. All plots share one y scale.
n_lines = HIGHLIGHT_MAX + len(TASKS) + 3
fig, axes = plt.subplots(2 * len(models), len(TARGETS), squeeze=False,
                         figsize=(6.5 * len(TARGETS), len(models) * (4 + .16 * n_lines)),
                         gridspec_kw={"height_ratios": [4, .16 * n_lines] * len(models)})
for k, model in enumerate(models):
    for j, lab in enumerate(TARGETS):
        ax, leg = axes[2 * k, j], axes[2 * k + 1, j]
        if (k, j) != (0, 0):
            ax.sharey(axes[0, 0])
        handles, title = plot_runs(ax, rows[(rows["phenotype"] == lab) & (rows["model"] == model)],
                                   base[lab][model]["std"])
        ax.set_title(f"{model} · {lab}")
        leg.axis("off")
        leg.legend(handles=handles, loc="upper center", fontsize=7, title=title, title_fontsize=7)
    axes[2 * k, 0].set_ylabel("gain over Age + Sex\n(Δ CV Spearman)")
fig.suptitle("every row, against its own model's covariates-only baseline · coloured = clears it at "
             f"{' and '.join(TASKS)}, ± sd over seeds", fontsize=8, color=INK2, y=.995)
fig.tight_layout()
save(fig, "fig1_rankings")

In [ ]:
# Every row that clears its baseline at both tasks, by model: gain per task (best worst-task gain first).
pd.concat({(lab, m): (lambda d: d[d["entity"].isin(above_both(d))]
                      .pivot_table(index=["tree", "arm", "label"], columns="task", values="gain")
                      .reindex(columns=list(TASKS))
                      .pipe(lambda t: t.loc[t.min(axis=1).sort_values(ascending=False).index]))(
               rows[(rows["phenotype"] == lab) & (rows["model"] == m)])
           for lab in TARGETS for m in models}, names=["phenotype", "model"]).round(3)

## 1.2–1.6 What drives the score

One figure per axis of the grid (`FACTORS`), on the same rows as figure 1 (`FACTOR_TREES`,
bstm by default), on the same **gain** scale. Every row is a faint dot, put in the group for its
window, atlas, model and so on. Rest and movie sit side by side in each group. The marker is the
median, and the bar spans the middle half (25th–75th percentile). The band is the largest
baseline sd of the models shown.

A group is pooled over everything else in the grid. If one window size only ever ran with a
strong atlas, it inherits that atlas's scores, so read a gap between groups as a lead to check
in figure 1, not as an effect.

In [ ]:
NUMERIC_ORDER = {"window": "window_s", "K_": "K"}       # ordered by size; the rest by median gain,
                                                        # once per figure so both panels match

def levels(d, col):
    if col in NUMERIC_ORDER:
        return d.groupby(col)[NUMERIC_ORDER[col]].first().sort_values().index.tolist()
    return d.groupby(col)["gain"].median().sort_values(ascending=False).index.tolist()

def compare(ax, d, col, sd, lev):
    """One panel: every row grouped by `col`, rest and movie side by side."""
    d = d.dropna(subset=[col])
    rng = np.random.default_rng(0)
    draw_zero(ax, sd)
    for j, task in enumerate(TASKS):
        dx, c = (j - (len(TASKS) - 1) / 2) * .3, TASK_COLOUR[task]
        for i, v in enumerate(lev):
            s = d.loc[(d[col] == v) & (d["task"] == task), "gain"]
            if s.empty:
                continue
            ax.scatter(i + dx + rng.uniform(-.06, .06, len(s)), s, s=8, color=c, alpha=.25, lw=0)
            q1, q2, q3 = np.percentile(s, [25, 50, 75])
            ax.plot([i + dx] * 2, [q1, q3], color=c, lw=3, solid_capstyle="round", zorder=3)
            ax.plot(i + dx, q2, TASK_MARKER[task], ms=7, color=c, mec=SURFACE, mew=1, zorder=4)
    long = max(map(len, map(str, lev))) > 8
    ax.set_xticks(range(len(lev)), lev, rotation=30 if long else 0, ha="right" if long else "center")
    ax.set_xlim(-.6, len(lev) - .4)

pool = rows[rows["tree"].isin(FACTOR_TREES)]
for k, (col, name) in enumerate(FACTORS.items(), start=2):
    fig, axes = plt.subplots(1, len(TARGETS), squeeze=False, sharey=True,
                             figsize=(6.5 * len(TARGETS), 3.9))
    lev = levels(pool.dropna(subset=[col]), col)
    for ax, lab in zip(axes[0], TARGETS):
        compare(ax, pool[pool["phenotype"] == lab], col, max(b["std"] for b in base[lab].values()), lev)
        ax.set_title(lab)
    axes[0, 0].set_ylabel("gain over Age + Sex\n(Δ CV Spearman)")
    handles = [Line2D([], [], ls="none", marker=TASK_MARKER[t], ms=7, color=TASK_COLOUR[t],
                      label=f"{t}: median, bar = middle half") for t in TASKS]
    fig.legend(handles=handles + [Patch(color=MUTED, alpha=.35, label="baseline ± sd (0 = no gain)")],
               loc="upper right", ncol=3, fontsize=7, bbox_to_anchor=(1, 1.02))
    fig.suptitle(f"1.{k}  {name}  ·  {', '.join(FACTOR_TREES)} · models: {', '.join(models)}",
                 x=.01, ha="left", fontsize=10, color=INK, y=1.0)
    fig.tight_layout()
    save(fig, f"fig1_{k}_{col}")
    plt.show()

## 2. Volcano with an effect size on x

**Why a correlation volcano (`X_AXIS = "r"`) is a narrow V.** A correlation already divides out each
cell's noise. With n fixed, p depends on |r| alone, so every dot lands on one curve. A gene volcano puts
a *raw* effect on x (log2 fold change) and takes p from a test that uses each feature's own variance,
so two features with the same fold change can have very different p.

**Why the raw fold change is too wide.** A ratio of two near-zero means is mostly noise. A rare cell
that moves from 0.0004 to 0.0010 is a 1.3 log2 fold change and means nothing. `SHRINK` adds a pseudocount
to both means: `log2((elevated + c) / (Normal + c))`, where `c = SHRINK × the average cell`.
Common cells barely move, and rare cells are pulled toward 0. This is the same idea as the
shrunken fold changes in DESeq2. `SHRINK = 0` gives the raw ratio. Raising it narrows the cloud
toward a slope-like scale. p is unaffected: it comes from the t-test of the group term in a
regression on the covariates.

There is one panel for each (task, phenotype) pair. Each panel draws its `TOP_N` sets from its own ranking,
so the colours in different panels belong to different sets: read the legend under each panel.
Dots are `(state set, i->j)` cells, and filled dots with a dark ring survive BH-FDR over every cell in the panel.
The **dashed** line is that FDR cut (the p of the weakest surviving cell). The **dotted** line is p = 0.05 uncorrected.
The cells are compositional, because each subject's row sums to 1, so hits tend to come in sprays.
Read the strongest cell and treat its neighbours as the same effect.

In [ ]:
CASE = ORDINAL_LEVELS.index(CASE_FROM)
XLABEL = {"log2fc": f"log2 fold change  ({CASE_FROM}+ vs Normal, adjusted, shrink={SHRINK:g})",
          "slope": "Δ transition probability per HADS level  (adjusted)",
          "r": f"partial Spearman r  (| {', '.join(COVARIATES)})"}[X_AXIS]

def effects(A, y, cov):
    """Effect of the phenotype on every column of A, adjusted for cov. Returns (x, p)."""
    if X_AXIS == "r":
        return partial_spearman(A, y, cov)
    g = (y >= CASE).astype(float) if X_AXIS == "log2fc" else y
    X = np.column_stack([g, design(cov)])
    B = np.linalg.lstsq(X, A, rcond=None)[0]
    df = len(A) - X.shape[1]
    se = np.sqrt(((A - X @ B) ** 2).sum(0) / df * np.linalg.pinv(X.T @ X)[0, 0])
    p = 2 * stats.t.sf(np.abs(B[0] / se), df)
    if X_AXIS == "slope":
        return B[0], p
    mu0 = A[g == 0].mean(0)                           # Normal
    mu1 = np.clip(mu0 + B[0], 0, None)                # elevated, adjusted; a mean cannot go below 0
    c = max(SHRINK * A.mean(), 1e-12)                 # A.mean() = the average cell
    return np.log2((mu1 + c) / (mu0 + c)), p

def scan(task, lab):
    """Every transition cell of the top sets vs one phenotype. One row per cell."""
    out = []
    for rank, s in top_sets(task, lab).iterrows():
        d, cols = merged(s, task, lab)
        A = d[cols]
        A = A.loc[:, (A > 0).mean() >= MIN_PRESENT]   # rare cells: a ratio of noise
        x, p = effects(A.to_numpy(float), d[lab].to_numpy(float), d[COVARIATES])
        out.append(pd.DataFrame({"rank": rank + 1, "set": s["label"], "score": s["mean"],
                                 "transition": A.columns, "x": x, "p": p, "n": len(d)}))
    v = pd.concat(out, ignore_index=True)
    v["p_fdr"] = stats.false_discovery_control(v["p"], method="bh")
    return v.assign(task=task, phenotype=lab)

volc = pd.concat([scan(task, lab) for task in TASKS for lab in TARGETS], ignore_index=True)
volc.nsmallest(10, "p")[["task", "phenotype", "rank", "set", "transition", "x", "p", "p_fdr", "n"]]

In [ ]:
def plot_volcano(ax, v):
    sig = v["p_fdr"] < ALPHA
    for k, (rank, g) in enumerate(v.groupby("rank")):
        c, m = SET_COLOURS[k % len(SET_COLOURS)], SET_MARKERS[k % len(SET_MARKERS)]
        y = -np.log10(g["p"])
        ax.scatter(g["x"], y, s=14, marker=m, color=c, alpha=.45, lw=0,
                   label=f"#{rank}  {g['set'].iloc[0]}")
        hit = sig[g.index]
        ax.scatter(g["x"][hit], y[hit], s=40, marker=m, color=c, edgecolor=INK, lw=.8)
    for t in v.nsmallest(LABEL_TOP, "p").itertuples():
        ax.annotate(f"{t.transition} (#{t.rank})", (t.x, -np.log10(t.p)), xytext=(4, 3),
                    textcoords="offset points", fontsize=7, color=INK2)
    ax.axhline(-np.log10(.05), color=MUTED, ls=":", lw=.8)
    if sig.any():
        ax.axhline(-np.log10(v.loc[sig, "p"].max()), color=INK2, ls="--", lw=.8)
    ax.axvline(0, color=MUTED, lw=.6)
    lim = 1.08 * v["x"].abs().max()
    ax.set_xlim(-lim, lim)
    ax.set_title(f"{v['task'].iloc[0]} · {v['phenotype'].iloc[0]}   "
                 f"({sig.sum()} / {len(v):,} cells FDR<{ALPHA:g}, n≈{int(v['n'].median())})",
                 fontsize=10)
    ax.legend(loc="upper center", bbox_to_anchor=(.5, -.16), ncol=2, fontsize=7,
              handletextpad=.3, columnspacing=1, markerscale=1.3)

fig, axes = plt.subplots(len(TASKS), len(TARGETS), squeeze=False,
                         figsize=(6.5 * len(TARGETS), 6.3 * len(TASKS)))
for (i, task), (j, lab) in ((a, b) for a in enumerate(TASKS) for b in enumerate(TARGETS)):
    plot_volcano(axes[i, j], volc[(volc["task"] == task) & (volc["phenotype"] == lab)])
for ax in axes[:, 0]:
    ax.set_ylabel(r"$-\log_{10}$ p")
for ax in axes[-1]:
    ax.set_xlabel(XLABEL)
fig.tight_layout(h_pad=3)
save(fig, f"fig2_volcano_{X_AXIS}")

### Editing notes

* **Figure 1's rows:** `FIG1_MODELS` / `FIG1_ARMS` narrow it. `MARGIN_SD = 1` asks a row's gain to clear
  its baseline by a whole sd of its own as well. `HIGHLIGHT_MAX` sets how many are named per panel.
* **Baseline:** each row of the runs also carries `n_transitions` (bstm) or the motion columns (fcm)
  on top of its features, so the gap to an Age + Sex line includes what those add. The baselines are
  fitted once, in their own cell, so re-drawing a figure does not refit them.
* **Figures 1.2–1.6:** add `"K"` or `"embedding"` to `FACTORS` (as `"K_"` for K), or drop an entry.
* **Figure 2:** `MODEL` / `ARM` pick its sets. With `ARM["bstm"] = "cells [cond]"`, the volcano tests
  the Markov p(j | i) instead of the stored p_joint. `X_AXIS = "r"` gives the correlation volcano.
* **More or fewer sets:** `TOP_N`. Past ten, colours repeat with a new marker shape.
* **A single panel:** call `plot_volcano(ax, volc[(volc.task == "rest") & (volc.phenotype == "anxiety")])`.
* **Hits for one set:** `volc.query("rank == 1 and phenotype == 'anxiety'").nsmallest(10, "p")`.
* `top_sets(task, phenotype)` shows exactly which rows of `summary.csv` the volcano drew from.
* The ranking score and the volcano answer different questions. A set can rank first because its signal
  is spread across many cells and still show no single cell above the FDR line.